# Factor Model Preparation

This notebook builds the minimal public factor-model preparation tables:

- `weekly_collection_panel_filtered`
- `weekly_returns_vw`

In [ ]:
import bigframes.pandas as bpd

PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
BASE_DATASET_ID = "<YOUR_BASE_BIGQUERY_DATASET>"
ANALYTICS_DATASET_ID = "<YOUR_ANALYTICS_BIGQUERY_DATASET>"

def fqn(dataset_id: str, table_name: str) -> str:
    return f"{PROJECT_ID}.{dataset_id}.{table_name}"

NFT_TRADING_USD_FILTERED_TABLE = fqn(BASE_DATASET_ID, "nft_trading_usd_filtered")
USD_ETH_BASE_TABLE = fqn(BASE_DATASET_ID, "usd_eth_base")
WEEKLY_COLLECTION_PANEL_TABLE = fqn(ANALYTICS_DATASET_ID, "weekly_collection_panel_filtered")
WEEKLY_RETURNS_TABLE = fqn(ANALYTICS_DATASET_ID, "weekly_returns_vw")

def is_exist_table(table_id: str) -> bool:
    try:
        bpd.read_gbq(f"SELECT 1 FROM `{table_id}` LIMIT 1")
        return True
    except Exception:
        return False

In [ ]:
def build_weekly_collection_panel_filtered(force_create: bool = False) -> None:
    table_id = WEEKLY_COLLECTION_PANEL_TABLE
    src_table = NFT_TRADING_USD_FILTERED_TABLE

    if is_exist_table(table_id) and not force_create:
        print(f"table: {table_id} exists, skip.")
        return

    sql = f"""
    CREATE OR REPLACE TABLE `{table_id}` AS
    SELECT
      collection,
      week_start,
      APPROX_QUANTILES(SAFE_CAST(price_usd AS FLOAT64), 100)[OFFSET(50)] AS median_price_usd,
      SUM(SAFE_CAST(price_usd AS FLOAT64)) AS total_volume_usd,
      COUNT(*) AS tx_count
    FROM `{src_table}`
    WHERE price_usd IS NOT NULL
      AND SAFE_CAST(price_usd AS FLOAT64) > 0
      AND collection IS NOT NULL
      AND week_start IS NOT NULL
    GROUP BY collection, week_start
    ORDER BY collection, week_start
    """
    _ = bpd.read_gbq(sql)
    print(f"Built: {table_id}")

In [ ]:
def build_weekly_returns_vw(force_create: bool = False) -> None:
    table_id = WEEKLY_RETURNS_TABLE
    nft_table = WEEKLY_COLLECTION_PANEL_TABLE
    fx_base_table = USD_ETH_BASE_TABLE

    if is_exist_table(table_id) and not force_create:
        print(f"table: {table_id} exists, skip.")
        return

    sql = f"""
    CREATE OR REPLACE TABLE `{table_id}` AS
    WITH nft_ret AS (
      SELECT
        collection,
        week_start,
        total_volume_usd,
        CASE
          WHEN LAG(week_start) OVER (PARTITION BY collection ORDER BY week_start) IS NOT NULL
           AND DATE_DIFF(
                 week_start,
                 LAG(week_start) OVER (PARTITION BY collection ORDER BY week_start),
                 DAY
               ) = 7
           AND median_price_usd > 0
           AND LAG(median_price_usd) OVER (PARTITION BY collection ORDER BY week_start) > 0
          THEN LOG(
            SAFE_DIVIDE(
              median_price_usd,
              LAG(median_price_usd) OVER (PARTITION BY collection ORDER BY week_start)
            )
          )
          ELSE NULL
        END AS nft_return
      FROM `{nft_table}`
    ),
    market_ret AS (
      SELECT
        week_start,
        SAFE_DIVIDE(
          SUM(nft_return * total_volume_usd),
          SUM(total_volume_usd)
        ) AS market_return
      FROM nft_ret
      WHERE nft_return IS NOT NULL
        AND total_volume_usd IS NOT NULL
        AND total_volume_usd > 0
      GROUP BY week_start
    ),
    fx_weekly AS (
      SELECT
        week_start,
        AVG(SAFE_CAST(usd_eth_rate AS FLOAT64)) AS avg_usd_eth_rate
      FROM `{fx_base_table}`
      WHERE usd_eth_rate IS NOT NULL
      GROUP BY week_start
    ),
    fx_ret AS (
      SELECT
        week_start,
        LOG(
          SAFE_DIVIDE(
            avg_usd_eth_rate,
            LAG(avg_usd_eth_rate) OVER (ORDER BY week_start)
          )
        ) AS fx_return
      FROM fx_weekly
    )
    SELECT
      n.collection,
      n.week_start,
      n.nft_return,
      m.market_return,
      f.fx_return
    FROM nft_ret AS n
    LEFT JOIN market_ret AS m ON n.week_start = m.week_start
    LEFT JOIN fx_ret AS f ON n.week_start = f.week_start
    WHERE n.nft_return IS NOT NULL
    ORDER BY n.collection, n.week_start
    """
    _ = bpd.read_gbq(sql)
    print(f"Built: {table_id}")

In [ ]:
build_weekly_collection_panel_filtered(force_create=True)
build_weekly_returns_vw(force_create=True)